# Sinh dataset SFT tiếng Việt cho ngành F&B

Notebook này sinh từng mẫu **tuần tự** (1 request/lần) và **append ngay** vào file output để tránh mất dữ liệu nếu bị ngắt.

**Thứ tự chạy:**
1. Cài deps
2. Kiểm tra kết nối Azure OpenAI
3. Load taxonomy + seed
4. Vòng lặp sinh mẫu (chạy lại bao nhiêu lần cũng được, sẽ tự skip mẫu đã có)

## 1. Cài đặt thư viện

In [30]:
# %pip install -q openai python-dotenv tenacity tqdm

## 2. Kiểm tra kết nối Azure OpenAI

Luôn reload `.env` mới nhất bằng `override=True`. Gửi 1 prompt "Xin chào" để xác nhận deployment hoạt động.

In [31]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI

# Tìm .env: ưu tiên thư mục notebook, fallback repo root
ENV_PATH = Path.cwd() / ".env"
if not ENV_PATH.exists():
    ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH, override=True)
print(f"Loaded .env: {ENV_PATH}")

AZURE_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
AZURE_KEY = os.environ["AZURE_OPENAI_API_KEY"]
OPENAI_API_VERSION = os.environ.get("OPENAI_API_VERSION", "2024-08-01-preview")
OPENAI_MODEL = os.environ.get("JUDGE_MODEL", "md-gpt-5.4-mini")

client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_KEY,
    api_version=OPENAI_API_VERSION,
)

Loaded .env: d:\Github\mcs-train-content-model\.env


In [33]:
_resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Xin chào, hãy trả lời ngắn gọn bằng tiếng Việt."}],
)
print(f"Endpoint   : {AZURE_ENDPOINT}")
print(f"Deployment : {OPENAI_MODEL}")
print(f"Api version: {OPENAI_API_VERSION}")
print(f"Reply      : {_resp.choices[0].message.content.strip()}")
print("\n✅ Kết nối Azure OpenAI thành công.")
 

Endpoint   : https://vqnhan-poc.openai.azure.com
Deployment : gpt-5.4
Api version: 2024-12-01-preview
Reply      : Xin chào! Tôi sẽ trả lời ngắn gọn bằng tiếng Việt.

✅ Kết nối Azure OpenAI thành công.


## 3. Cấu hình taxonomy F&B + load seed

In [34]:
import json
import random

SCRIPT_DIR = Path.cwd()
OUTPUT_FILE = SCRIPT_DIR / "fnb_dataset_train.json"

TARGET_COUNT = 100  # tổng số mẫu mong muốn trong OUTPUT_FILE

# Khi IS_TEST=True: chỉ sinh instruction + title + seed (không gọi LLM để viết content).
# Dùng để tạo eval cases — model sẽ tự sinh content khi đánh giá.
# Khi IS_TEST=False: sinh đủ 4 field (instruction + title + seed + content) cho SFT training.
IS_TEST = False

SUB_SEGMENTS = [
    "Chuỗi cà phê tầm trung", "Cà phê đặc sản specialty", "Cà phê take-away",
    "Trà sữa", "Trà trái cây / matcha", "Nước ép & smoothie healthy",
    "Nhà hàng fine-dining", "Nhà hàng buffet", "Chuỗi lẩu nướng BBQ",
    "Quán nhậu / beer club", "Bia thủ công craft beer", "Rượu vang nhập khẩu",
    "Fast food gà rán / burger", "Pizza chuỗi", "Tiệm bánh ngọt bakery",
    "Kem / dessert shop", "Cloud kitchen giao hàng", "Chuỗi bún phở",
    "Cơm văn phòng / cơm tấm", "Đồ ăn vặt street food",
    "Snack đóng gói (bánh kẹo)", "Sản phẩm sữa / sữa chua",
    "Gia vị / nước chấm đóng chai", "Thực phẩm đông lạnh tiện lợi",
    "Đặc sản OCOP vùng miền", "Siêu thị mini / grocery",
    "Đồ uống đóng chai RTD", "Trà thảo mộc / nước detox",
    "Nhà hàng chay / thuần chay", "Cửa hàng tiện lợi 24/7",
]

WORKFLOW_STAGES = [
    "Chiến lược thương hiệu", "Định vị & naming sản phẩm mới",
    "Menu engineering & pricing", "Kế hoạch ra mắt sản phẩm mới",
    "Content calendar mạng xã hội", "Caption Facebook",
    "Caption Instagram", "Kịch bản video TikTok",
    "Kế hoạch livestream TikTok Shop", "Kế hoạch livestream Facebook",
    "Chiến lược KOL/KOC foodie", "Mời food reviewer & sự kiện trải nghiệm",
    "Quảng cáo Facebook Ads", "Quảng cáo TikTok Ads",
    "Quảng cáo Google Ads", "Quảng cáo GDN / banner",
    "Bài blog SEO", "Landing page khuyến mãi",
    "Email marketing", "SMS / Zalo OA broadcast",
    "Chương trình loyalty / membership", "Push notification app",
    "Khai trương cửa hàng mới", "Sự kiện activation tại điểm bán",
    "Chiến dịch Tết Nguyên Đán", "Chiến dịch Trung Thu",
    "Chiến dịch Valentine / 8-3 / 20-10", "Chiến dịch Giáng Sinh / Năm Mới",
    "Black Friday / Shopee 11-11 sale", "CSR & sustainability",
    "Crisis PR sự cố an toàn thực phẩm", "Phản hồi review tiêu cực",
    "Pitch dịch vụ B2B catering", "Tuyển dụng nhượng quyền franchise",
    "Employer branding tuyển bếp/barista", "A/B testing creative",
    "Báo cáo & phân tích hiệu quả", "Audience research & insight",
    "Packaging design brief", "OOH billboard ngã tư",
]

LENGTH_PROFILES = [
    ("ngắn gọn 50-100 từ", "Response súc tích, đi thẳng vào ý chính, không lan man."),
    ("trung bình 120-200 từ", "Response chi tiết vừa phải, có cấu trúc rõ ràng."),
    ("dài 250-400 từ", "Response chuyên sâu, đầy đủ KPI, phân bổ ngân sách, đo lường cụ thể."),
]

TONE_VARIANTS = [
    "chuyên nghiệp như agency",
    "thân thiện như freelancer",
    "kiểu founder startup",
    "kiểu director marketing dày dạn",
]

print(f"✅ Sẵn sàng sinh mẫu. IS_TEST={IS_TEST} | OUTPUT_FILE={OUTPUT_FILE.name}")

✅ Sẵn sàng sinh mẫu. IS_TEST=False | OUTPUT_FILE=fnb_dataset_train.json


## 4. Prompt template + hàm tiện ích

In [35]:
import hashlib
import re
from tenacity import retry, stop_after_attempt, wait_exponential

# Schema khi IS_TEST=False (SFT training): instruction + title + seed + content
# Schema khi IS_TEST=True  (eval cases):   instruction + title + seed
#
# instruction : system-level copywriter directive — giọng, phong cách, ràng buộc output
# title       : 1 câu ngắn mô tả nhiệm vụ cụ thể (= input_title khi eval)
# seed        : data mồi có cấu trúc 7 trường nhãn:giá trị (= seed_content khi eval)
# content     : bài Facebook hoàn chỉnh (= actual_output khi eval, bỏ qua nếu IS_TEST)

_SCHEMA_TRAINING = '{"instruction": "...", "title": "...", "seed": "...", "content": "..."}'
_SCHEMA_TEST     = '{"instruction": "...", "title": "...", "seed": "..."}'

SYSTEM_PROMPT = """Bạn là chuyên gia marketing F&B Việt Nam với 10+ năm kinh nghiệm, đã làm cho Highlands, Phúc Long, Golden Gate, Masan. Bạn đang viết dữ liệu training SFT cho agent tạo nội dung Facebook marketing F&B.

YÊU CẦU TUYỆT ĐỐI:
1. Trả về DUY NHẤT 1 đối tượng JSON hợp lệ, không markdown, không giải thích, không ```json
2. Tiếng Việt tự nhiên, dùng thuật ngữ marketing chuẩn (CPL, CTR, ROAS, GMV, AOV, retention...)
3. "instruction": 1-2 câu system directive cho copywriter — giọng viết, phong cách, ràng buộc output. Ví dụ: "Bạn là copywriter marketing người Việt, giọng founder F&B, viết Facebook-native, đoạn ngắn, CTA cụ thể, KHÔNG hype rỗng."
4. "title": 1 câu ngắn mô tả nhiệm vụ cụ thể — loại bài + tên thương hiệu/sản phẩm + hook chính. Ví dụ: "Viết caption Facebook cho deal cuối tuần của Moon Oven Bakery — Signature Weekend Box giảm 25%."
5. "seed": DATA MỒI CÓ CẤU TRÚC với đúng 7 trường dán nhãn theo dạng "Nhãn: giá trị", mỗi trường 1 dòng:
   - Công ty / Thương hiệu: tên thật + mô tả định vị ngắn (ví dụ: "Moon Oven Bakery — tiệm bánh artisan phong cách Âu, 5 cửa hàng TP.HCM")
   - Sản phẩm: tên sản phẩm + thông số cụ thể (trọng lượng, dung tích, thành phần chính, số lượng...)
   - Giá / Ưu đãi: con số VND cụ thể + giá gốc nếu có, % giảm, combo, freeship, quà tặng kèm
   - Điều kiện: thời gian áp dụng, platform (MoMo / GrabFood / inbox...), phạm vi giao hàng, giới hạn số lượng
   - Đối tượng: nhân khẩu học cụ thể (tuổi, nghề nghiệp, vị trí địa lý)
   - Kênh: Facebook Page, inbox Messenger, ShopeeFood, TikTok Shop, Zalo OA...
   - Mục tiêu KPI: 1-2 KPI với con số cụ thể (CTR ≥ x%, CPL < xk, GMV ≥ x triệu, ROAS ≥ x...)
   Seed phải đủ factual để agent viết bài mà không cần bịa thêm claim.

VÍ DỤ SEED TỐT (đúng chuẩn — 7 trường đầy đủ, có số cụ thể):
Công ty / Thương hiệu: Moon Oven Bakery — tiệm bánh ngọt artisan phong cách Âu, 5 cửa hàng tại TP.HCM
Sản phẩm: Signature Weekend Box — combo 6 bánh ngọt (tổng 420g), gói hộp thiết kế "Weekend Edition"
Giá / Ưu đãi: 149.000đ (giảm 25% từ giá gốc 199.000đ), freeship trong 3km
Điều kiện: Áp dụng thứ 7 và chủ nhật, đặt qua app MoMo hoặc nhắn inbox Facebook Page, giới hạn 50 hộp/ngày
Đối tượng: Người trẻ 22–35 tuổi sống và làm việc trong bán kính 3km tại TP.HCM
Kênh: Facebook Page Moon Oven Bakery, inbox Messenger, app MoMo
Mục tiêu KPI: CTR ≥ 3%, GMV cuối tuần ≥ 8 triệu đồng

6. "content" (chỉ khi được yêu cầu): bài Facebook hoàn chỉnh, copy/paste ready, có hook, nội dung chính, CTA, bám sát seed. KHÔNG bịa claim không có trong seed.
7. Bối cảnh Việt Nam thật: tên thương hiệu/KOL/địa danh/nền tảng VN (Shopee, TikTok Shop, Zalo, ShopeeFood, GrabFood, Momo, ZaloPay, Sơn Tùng, Trấn Thành, Khoai Lang Thang, Hà Linh...)
8. Số tiền dùng VND (triệu/tỷ), không dùng USD"""

_CONTENT_FIELD_DESC = "\n- content: bài Facebook hoàn chỉnh, copy/paste lên Facebook được ngay, bám sát seed."


def _build_user_message(
    segment: str, workflow: str, length: str, tone: str, length_note: str, is_test: bool
) -> str:
    """Build the full user message with an f-string — avoids .format() choking on JSON braces."""
    schema = _SCHEMA_TEST if is_test else _SCHEMA_TRAINING
    content_desc = "" if is_test else _CONTENT_FIELD_DESC
    return (
        f"Tạo 1 mẫu marketing F&B Việt Nam cho content agent Facebook với cấu hình:\n\n"
        f"- Phân khúc F&B: {segment}\n"
        f"- Giai đoạn workflow: {workflow}\n"
        f"- Độ dài content: {length}\n"
        f"- Tông giọng: {tone}\n\n"
        f"{length_note}\n\n"
        f"Trả về DUY NHẤT 1 JSON object với schema:\n"
        f"{schema}\n\n"
        f"Mô tả từng field:\n"
        f"- instruction: 1-2 câu system directive — giọng, phong cách, ràng buộc output cho copywriter.\n"
        f"- title: 1 câu ngắn — loại bài Facebook + tên thương hiệu/sản phẩm + hook chính.\n"
        f"- seed: data mồi có cấu trúc, đúng 7 trường (Công ty / Thương hiệu, Sản phẩm, Giá / Ưu đãi, "
        f"Điều kiện, Đối tượng, Kênh, Mục tiêu KPI) — mỗi trường 1 dòng \"Nhãn: giá trị\", có số cụ thể.{content_desc}\n\n"
        f"Không giải thích, không thêm text ngoài JSON."
    )


def sample_hash(sample: dict) -> str:
    text = sample.get("title", "") + sample.get("seed", "")[:200]
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def extract_json(text: str) -> dict:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"Không tìm thấy JSON: {text[:200]}")
    return json.loads(text[start : end + 1])


def validate(sample: dict, is_test: bool) -> dict:
    if not isinstance(sample, dict):
        raise ValueError("Không phải dict")
    required_keys = ("instruction", "title", "seed") if is_test else ("instruction", "title", "seed", "content")
    for key in required_keys:
        if key not in sample or not isinstance(sample[key], str):
            raise ValueError(f"Thiếu/sai field: {key}")
        if len(sample[key].strip()) < 10:
            raise ValueError(f"Field {key} quá ngắn")
    seed_lines = [l for l in sample["seed"].splitlines() if ":" in l and len(l.strip()) > 5]
    if len(seed_lines) < 4:
        raise ValueError(f"seed thiếu cấu trúc: chỉ có {len(seed_lines)} dòng nhãn:giá trị, cần ≥ 4")
    return {k: sample[k].strip() for k in required_keys}


@retry(stop=stop_after_attempt(1), wait=wait_exponential(multiplier=2, min=2, max=15))
def generate_one(segment: str, workflow: str, is_test: bool = IS_TEST) -> dict:
    length_label, length_note = random.choice(LENGTH_PROFILES)
    tone = random.choice(TONE_VARIANTS)
    user_msg = _build_user_message(
        segment=segment,
        workflow=workflow,
        length=length_label,
        tone=tone,
        length_note=length_note,
        is_test=is_test,
    )
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.9,
        top_p=0.95,
    )
    return validate(extract_json(resp.choices[0].message.content), is_test)


print("✅ Đã định nghĩa hàm sinh mẫu.")

✅ Đã định nghĩa hàm sinh mẫu.


In [36]:
# Test generate_one với 1 mẫu
try:
    test_sample = generate_one("Trà sữa", "Caption Facebook")
    print(f"✅ Test thành công! (IS_TEST={IS_TEST})")
    print("Instruction:", test_sample["instruction"][:120])
    print("Title      :", test_sample["title"][:120])
    print("Seed       :", test_sample["seed"][:200])
    if "content" in test_sample:
        print("Content    :", test_sample["content"][:120])
except Exception as e:
    print(f"❌ Lỗi test: {e}")
    import traceback
    traceback.print_exc()

✅ Test thành công! (IS_TEST=False)
Instruction: Bạn là copywriter marketing người Việt cho ngành F&B, viết caption Facebook-native giọng chuyên nghiệp như agency: rõ lợ
Title      : Viết caption Facebook cho deal trà sữa mùa hè của Lộc Tea House — combo 2 ly Trân Châu Đường Đen giảm giá, chốt đơn qua 
Seed       : Công ty / Thương hiệu: Lộc Tea House — chuỗi trà sữa phong cách hiện đại, 8 cửa hàng tại TP.HCM, định vị nguyên liệu chuẩn vị và giao nhanh nội thành
Sản phẩm: Combo 2 ly Trà Sữa Trân Châu Đường Đen s
Content    : 14:00-17:00 là lúc cần một ly ngọt vừa đủ để kéo mood làm việc lên lại — và đây là deal đáng chốt trong tuần này từ Lộc 


## 5. I/O an toàn — load output hiện có & append từng mẫu

Mỗi mẫu sinh ra sẽ được ghi vào file ngay lập tức (atomic write qua file tạm). Có thể dừng và chạy lại bất cứ lúc nào — script sẽ tiếp tục từ chỗ đang dở.

In [37]:
import tempfile


def load_output() -> list:
    if OUTPUT_FILE.exists():
        with OUTPUT_FILE.open("r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_output_atomic(data: list) -> None:
    """Ghi file qua tmp + rename để tránh corrupt khi bị ngắt giữa chừng."""
    fd, tmp_path = tempfile.mkstemp(
        dir=str(OUTPUT_FILE.parent), prefix=".tmp_", suffix=".json"
    )
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp_path, OUTPUT_FILE)
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise


current = load_output()
print(f"Output hiện có: {len(current)} mẫu trong {OUTPUT_FILE.name}")

Output hiện có: 0 mẫu trong fnb_dataset_train.json


## 6. Vòng lặp sinh mẫu (tuần tự, append từng mẫu)

Chạy lại cell này bao nhiêu lần cũng được. Mỗi mẫu xong sẽ flush vào file `fnb_dataset_vi.json` ngay.

In [38]:
# Sinh ma trận task đa dạng
task_matrix = [(s, w) for s in SUB_SEGMENTS for w in WORKFLOW_STAGES]
random.shuffle(task_matrix)
print("Example task:", task_matrix[0])

Example task: ('Trà sữa', 'Kế hoạch livestream Facebook')


In [39]:
from tqdm.auto import tqdm

samples = load_output()
seen_hashes = {sample_hash(s) for s in samples}

effective_target = TARGET_COUNT

remaining = effective_target - len(samples)
print(f"Hiện có: {len(samples)} | Mục tiêu: {effective_target} | Cần sinh thêm: {remaining}")

if remaining <= 0:
    print("✅ Đã đủ mẫu, không sinh thêm.")
else:
    pbar = tqdm(total=remaining, desc="Sinh mẫu")
    task_idx = 0
    error_count = 0
    while len(samples) < effective_target:
        seg, wf = task_matrix[task_idx % len(task_matrix)]
        task_idx += 1
        try:
            new_sample = generate_one(seg, wf)
            h = sample_hash(new_sample)
            if h in seen_hashes:
                pbar.set_postfix_str(f"trùng, skip ({seg[:15]})")
                continue
            seen_hashes.add(h)
            samples.append(new_sample)
            save_output_atomic(samples)  # flush ngay sau mỗi mẫu
            pbar.update(1)
            pbar.set_postfix_str(f"{seg[:18]} / {wf[:18]}")
            error_count = 0
        except Exception as e:
            error_count += 1
            pbar.write(f"⚠️ Lỗi ({seg[:20]} / {wf[:20]}): {str(e)[:120]}")
            if error_count >= 10:
                pbar.write("❌ Quá nhiều lỗi liên tiếp, dừng. Kiểm tra rate limit / quota.")
                break
    pbar.close()

print(f"\n✅ Tổng số mẫu trong {OUTPUT_FILE.name}: {len(samples)}")

Hiện có: 0 | Mục tiêu: 100 | Cần sinh thêm: 100


Sinh mẫu: 100%|██████████| 100/100 [23:07<00:00, 13.87s/it, Nhà hàng chay / th / Mời food reviewer ]


✅ Tổng số mẫu trong fnb_dataset_train.json: 100


## 7. (Tuỳ chọn) Xem thử 3 mẫu vừa sinh

In [40]:
data = load_output()
print(f"Tổng: {len(data)} mẫu | IS_TEST={IS_TEST}\n")
for i, s in enumerate(data[-3:], 1):
    print(f"=== Mẫu cuối #{i} ===")
    print("Instruction:", s["instruction"])
    print("Title      :", s["title"])
    print("Seed       :", s["seed"][:200], "..." if len(s["seed"]) > 200 else "")
    if "content" in s:
        print("Content    :", s["content"][:300], "..." if len(s["content"]) > 300 else "")
    print()

Tổng: 100 mẫu | IS_TEST=False

=== Mẫu cuối #1 ===
Instruction: Bạn là copywriter marketing người Việt, giọng director marketing F&B dày dạn, viết Facebook-native ngắn gọn 50-100 từ, ưu tiên logic giá trị món và CTA rõ ràng. Không hype rỗng, không bịa claim ngoài seed, câu ngắn, nhịp chắc.
Title      : Viết caption Facebook cho set giá mới của Lúa Roastery — Signature Pour Over Pairing, tối ưu AOV với combo dùng thử specialty.
Seed       : Công ty / Thương hiệu: Lúa Roastery — chuỗi cà phê specialty Việt, 3 cửa hàng tại TP.HCM, định vị hạt rang theo mùa và trải nghiệm brew thủ công
Sản phẩm: Signature Pour Over Pairing — 1 ly pour over  ...
Content    : Một mức giá dễ vào hơn để bắt đầu với specialty. Signature Pour Over Pairing gồm 1 pour over 250ml từ Arabica Đà Lạt + 1 financier bơ hạnh nhân 55g, nay còn 99.000đ thay vì 117.000đ. Nếu muốn đi tiếp, thêm cold brew bottle 300ml chỉ 45.000đ để tối ưu trải nghiệm và ngân sách. Áp dụng 20/05–02/06/202 ...

=== Mẫu cuối #2 ===
Instruction: